# Which values to steer with? GPT's value ranking and steering-vector similarity

Two questions for choosing values beyond toxicity:
1. **Relevance:** for each comment GPT picked the 3 most relevant values in each of VISPA's 5 categories. How often was
   each value picked?
2. **Distinctness:** how similar are the values' steering vectors (cosine, averaged over the steered layers 10-25)?
   Values close to toxicity (cosine ~0.5-0.6) steer Llama much like it, as harmfulness and fairness did.

In [9]:
RANKING = "../data/processed/civil-comments/consensus_n2000_train/value_ranking/gpt-6-luna_top3.jsonl"
VECTORS = "../data/processed/steering_vectors/Meta-Llama-3-8B-Instruct"
LAYERS = range(10, 26)   # the steered layers

In [10]:
import collections
import json

import numpy as np
import plotly.graph_objects as go
from sklearn.metrics.pairwise import cosine_similarity

from ethos.steering.activation import load_directions

# VISPA's categories: (name in the ranking file, short name, colour, values)
CATS = [("Schwartz's Basic Human Values", "Basic Human Values", "#3a7ca5",
         ["Self-Direction", "Stimulation", "Hedonism", "Achievement", "Power", "Security", "Conformity", "Tradition",
          "Benevolence", "Universalism"]),
        ("Cultural Dimensions", "Cultural Dimensions", "#e08a2e",
         ["Power Distance", "Uncertainty Avoidance", "Individualism", "Masculinity", "Long-Term Orientation",
          "Indulgence"]),
        ("Moral Theories", "Moral Theories", "#6a3d9a",
         ["Commonsense Morality", "Deontology", "Utilitarianism", "Justice", "Virtue Ethics", "Ubuntu", "Confucianism"]),
        ("AI Safety-Related Values", "AI Safety", "#d6559e", ["Fairness", "Truthfulness", "Toxicity", "Harmfulness"]),
        ("Non-WEIRD Moral Constructs", "Non-WEIRD Morals", "#2a9d8f", ["Face", "Karma", "Honor", "Spirituality"])]
VECTOR_NAME = {"Spirituality": "spiritual"}          # ranking name -> vector file name, where they differ
vector_name = lambda v: VECTOR_NAME.get(v, v.lower().replace(" ", "_").replace("long-term", "long_term"))

rows = [json.loads(line) for line in open(RANKING)]
counts = {key: collections.Counter(v for r in rows for v in r["selected"][key]) for key, *_ in CATS}
print(f"{len(rows)} comments, 3 values per category each")

2000 comments, 3 values per category each


## How often GPT picked each value

In [11]:
fig = go.Figure()
y, heads = 0, []
for key, name, col, values in reversed(CATS):                      # first category at the top
    vs = sorted(values, key=lambda v: counts[key][v])
    n = [counts[key][v] for v in vs]
    fig.add_bar(y=[y + i for i in range(len(vs))], x=n, orientation="h", marker_color=col, name=name,
                text=[f"{c:,}" for c in n], textposition="outside", customdata=vs,
                hovertemplate=f"<b>%{{customdata}}</b><br>{name}<br>%{{x:,}} of {len(rows):,} comments<extra></extra>")
    fig.add_annotation(x=0, y=y + len(vs) - 0.1, text=f"<b>{name}</b>", showarrow=False, xanchor="left",
                       yanchor="bottom", font=dict(color=col, size=13))
    heads.append(y + len(vs) + 0.75)
    y += len(vs) + 1.5
ticks = [(t, v) for tr in fig.data for t, v in zip(tr.y, tr.customdata)]
for h in heads[:-1]:
    fig.add_shape(type="line", xref="paper", x0=-0.35, x1=1, y0=h, y1=h, line=dict(color="#999", width=1, dash="dot"))
fig.update_yaxes(tickvals=[t for t, _ in ticks], ticktext=[v for _, v in ticks], range=[-0.7, y - 1.0], showgrid=False)
fig.update_xaxes(range=[0, 2250], title=f"Comments Where GPT Selected the Value (of {len(rows):,})", gridcolor="#eee")
fig.update_layout(height=780, width=650, showlegend=False, plot_bgcolor="white", bargap=0.25,
                  margin=dict(l=10, r=20, t=20, b=50), font=dict(size=12))
fig.show()

## Similarity of the steering vectors: GPT's top values per category

Cosine similarity between the steering vectors of GPT's `k` most-picked values in each category (`k` = 3, 2, 1).

In [12]:
def similarity(values):
    """Cosine similarity matrix of the values' steering vectors, averaged over LAYERS."""
    V = [load_directions(f"{VECTORS}/{vector_name(v)}.npz") for v in values]
    return np.mean([cosine_similarity([d[l] for d in V]) for l in LAYERS], axis=0)

def top(key, values, k):
    return sorted(values, key=lambda v: -counts[key][v])[:k]

def plot_similarity(k):
    groups = [(name, top(key, values, k)) for key, name, _, values in CATS]
    values = [v for _, vs in groups for v in vs]
    C = similarity(values)
    labels = [f"<span style='color:#d6559e'>{v}</span>" if v == "Toxicity" else v for v in values]
    fig = go.Figure(go.Heatmap(z=C, x=list(range(len(values))), y=list(range(len(values))), zmin=-1, zmax=1,
                               colorscale="RdBu_r", text=np.round(C, 2), texttemplate="%{text:.2f}",
                               textfont=dict(size=10), customdata=np.array([[(a, b) for b in values] for a in values]),
                               hovertemplate="%{customdata[0]} vs %{customdata[1]}: %{z:.2f}<extra></extra>",
                               colorbar=dict(title="Cosine<br>Similarity")))
    if k > 1:
        for b in range(k, len(values), k):
            for kw in (dict(x0=-0.5, x1=len(values) - 0.5, y0=b - 0.5, y1=b - 0.5),
                       dict(y0=-0.5, y1=len(values) - 0.5, x0=b - 0.5, x1=b - 0.5)):
                fig.add_shape(type="line", line=dict(color="#222", width=2), **kw)
    for g, (name, _) in enumerate(groups):
        mid = k * g + (k - 1) / 2
        fig.add_annotation(xref="paper", x=-0.36 if k > 1 else -0.62, y=mid, text=name.replace(" ", "<br>", 1),
                           showarrow=False, xanchor="right", font=dict(color="#555", size=12))
        if k > 1:
            fig.add_annotation(x=mid, yref="paper", y=1.0, text=name.replace(" ", "<br>", 1), showarrow=False,
                               yanchor="bottom", font=dict(color="#555", size=12))
    fig.update_xaxes(tickvals=list(range(len(values))), ticktext=labels, tickangle=-45, showgrid=False)
    fig.update_yaxes(tickvals=list(range(len(values))), ticktext=labels, autorange="reversed", showgrid=False,
                     scaleanchor="x")
    size = {3: 820, 2: 680, 1: 520}[k]
    fig.update_layout(width=size + 120, height=size, plot_bgcolor="white", margin=dict(l=230, r=20, t=70, b=20),
                      font=dict(size=12))
    fig.show()

In [13]:
plot_similarity(2)